In [1]:
t = 'GCTAGCTCTACGAGTCTA'
p = 'TCTA'

In [2]:
def bm_naive(p,t):
    occurances = []
    i = 0
    while i < len(t) - len(p) +1:
        matched = True
        for j in range(len(p) -1,-1,-1):
            if p[j]!=t[i+j]:
                matched = False
                break
        if matched:
            occurances.append(i)
        i += 1
    return occurances
            
            

In [3]:
def bad_character_shift(p,j,c):
    #look left of the position j in p for character c
    #walk from j-1 to 0
    for k in range(j-1,-1,-1 ):
        if p[k]==c:
            #Found it at a k shift so p[k] lines up with mismatch
            return j-k
    #c doesn't appear in the p
    return j+1
            


In [4]:
def boyer_moore_bad_char(p,t):
    occurrences = []
    i=0
    while i < len(t) - len(p) + 1:
        shift = 1
        mismatched = False
        for j in range(len(p)-1,-1,-1):
            if p[j] != t[i+j]:
                shift = bad_character_shift(p,j,t[i+j])
                mismatched = True
                break
        if not mismatched:
            occurrences.append(i)
        i += shift
    return occurrences
                

In [5]:
def good_suffix_shift(p,j):
    #mismatch at position j in p
    # the good suffix is everything after j : p[j+1:]
    good_suffix = p[j+1:]

    # CASE A: look for another occurrence of good_suffix earlier in p[:j+1]
    # We want the rightmost earlier occurrence (smallest shift).
    # So we search starting from the right side of p[:j+1] going left.

    # The earlier copy can start anywhere from position 0 up to position j.
    # We walk k from j down to 0 and check if good_suffix starts at p[k].

    for k in range(j,-1,-1):
        if p[k:k+len(good_suffix)] == good_suffix:
            #found an earlier copy of starting at position k
            #shift = j - k +1
            return j - k + 1
    #case A didn't find anything
    #case B: Longets prefix of p that matches a suffix of a good suffix
    #Try prefix length from longest to shrotest  longest is the smallest shift
    for l in range(len(good_suffix),0,-1):
        if p[:l] == good_suffix[-l:]:
            return len(p) - l
    #case C: no overlap at all shift past through the good suffix entirely
    return len(p) 
        
    
    

In [6]:
# Full test of good_suffix_shift across all three cases

print("Case A — full earlier copy of good suffix exists in P:")
print(good_suffix_shift('ABCDABC', 3))    # expect 4
print(good_suffix_shift('GCAGAGAG', 5))   # expect 2
print(good_suffix_shift('CTTACTT', 3))    # expect 4

print("\nCase B — partial overlap (prefix of P matches suffix of good suffix):")
print(good_suffix_shift('ABXXXYZAB', 4))  # expect 7

print("\nCase C — no overlap, shift past entirely:")
print(good_suffix_shift('XYZWAB', 3))     # expect 6

Case A — full earlier copy of good suffix exists in P:
4
2
4

Case B — partial overlap (prefix of P matches suffix of good suffix):
7

Case C — no overlap, shift past entirely:
6


In [12]:
def match_skip(p):
    """
    After a full match of p, return the shift amount.
    
    This is the good suffix rule applied to the entire pattern:
      - Case B analog: longest proper prefix of p that matches a suffix of p
      - Case C analog: no such prefix exists, shift past entirely
    
    (Case A analog — full P appearing inside itself — is impossible
    for a proper sub-occurrence, so we skip it.)

    """
    # case B : longetst l < len(p) where  p[-l:] == p[:l] 
    for l in range(len(p)-1,0,-1):
        if p[:l]==p[-l:]:
            return len(p)  - l
    #case C : Shift past entirely
    return len(p)


In [13]:
def boyer_moore_full(p, t):
    occurrences = []
    i = 0
    while i < len(t) - len(p) + 1:
        shift = 1
        mismatched = False
        for j in range(len(p) - 1, -1, -1):
            if p[j] != t[i + j]:
                # Mismatch — use both rules, take the largest safe shift
                bc = bad_character_shift(p, j, t[i + j])
                gs = good_suffix_shift(p, j)
                shift = max(shift, bc, gs)
                mismatched = True
                break
        if not mismatched:
            # Full match — record it, then use match_skip
            occurrences.append(i)
            shift = max(shift, match_skip(p))
        i += shift
    return occurrences

In [14]:
# Test 1: standard case
t = 'GCTAGCTCTACGAGTCTA'
p = 'TCTA'
print(boyer_moore_full(p, t))    # expect [6, 14]

# Test 2: pattern with self-overlap (where match_skip earns its keep)
t2 = 'ABCABCABCABC'
p2 = 'ABCABC'
print(boyer_moore_full(p2, t2))  # expect [0, 3, 6]

# Test 3: confirm consistency with simpler versions
print(bm_naive(p, t))                # expect [6, 14]
print(boyer_moore_bad_char(p, t))    # expect [6, 14]

[6, 14]
[0, 3, 6]
[6, 14]
[6, 14]
